# LongFlow P1 -- shaky-voice ablation (isolate the cause before building more)

Runtime: **L4 GPU**. ~20-30 min, ~$1-2 (mostly one new 5K-step training run;
the other two checkpoints are reused if they already exist on Drive).

Josh's ear caught a consistent ~1-2 Hz "good/degraded" quality wobble on
EVERY flow4/flow16 sample from the gate v2 checkpoint, roughly equally on
short and long bins, with barely any NFE-4-vs-16 difference (already rules
out sampling resolution as the primary cause). Two live hypotheses:
undertraining (5K steps, generic to this recipe) vs. sigma-mixing
(training on v2's clean+noised frames undifferentiated, no conditioning
to tell them apart -- exactly the gap the project's own plan already
flagged before capture v2 was built).

Three checkpoints, same architecture/hyperparameters/step count (5000
steps, batch 1024, lr 2e-4, ema 0.999) -- ONLY the data pool differs:

| tag | data pool | reused if exists? |
|---|---|---|
| **A** `gate_v2_5k.pt` | full v2 cache, clean+noised mixed (already have this) | yes, no retrain |
| **B** `gate_v2_clean5k.pt` | v2 cache, sigma==0 frames only | new, this is the one real training run |
| **C** `gate_5k.pt` | the original v1 cache (short clips, always clean) | yes if it survived from the July gate, else retrain |

All three then render the SAME short + long test utterances at NFE 4, for
a direct A/B/C listen. If B is clean and A is shaky -> sigma-mixing
confirmed. If both B and C are shaky -> generic undertraining, capture v2
is not implicated. If all three are shaky including C (a completely
different, all-clean, short-clip-only cache) -> something else entirely,
worth a fresh look before assuming anything.

Pre-registered: `experiments/p1_flow_head/NOTES.md` ("GATE V2" close-out
entry, watch item).

In [ ]:
# ===== COLD START (idempotent) -- run me first, wait for READY =====
NOTEBOOK_VERSION = "Shaky-voice ablation v1.0 (2026-08-17)"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, sys, time
import soundfile as sf
import numpy as np
from IPython.display import Audio, display

CACHE_V2_DIR = "/content/drive/MyDrive/longflow_p1_cache_v2"
CACHE_V1_DIR = "/content/drive/MyDrive/longflow_p1_cache"
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
assert os.path.exists(CACHE_V2_DIR) and os.path.exists(CACHE_V1_DIR), "both caches must exist on Drive"

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed -- check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import load_utterance
from src.flow_head.model import FlowHead, FlowHeadConfig
from src.flow_head.trainer import PairData, save_checkpoint, load_checkpoint, train, sample_latents

def decode_latents(z, chunk_frames=225):  # 225 frames = 30s @ 7.5Hz, avoids the OOM
    # hit on capture v2's long samples (fixed 2026-08-16/17, see gate_v2_colab.ipynb)
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    wavs, shape_fn = [], None
    for i in range(0, z.shape[0], chunk_frames):
        chunk = z[i : i + chunk_frames]
        candidates = [shape_fn] if shape_fn is not None else [
            lambda c: c.unsqueeze(0), lambda c: c.unsqueeze(0).transpose(1, 2)
        ]
        decoded = None
        for fn in candidates:
            try:
                out = model.model.acoustic_tokenizer.decode(fn(chunk))
                decoded = out[0] if isinstance(out, tuple) else out
                shape_fn = fn
                break
            except Exception as e:
                print(f"decode attempt {tuple(fn(chunk).shape)} failed: {repr(e)[:150]}")
        if decoded is None:
            raise RuntimeError("both decode shapes failed -- paste the errors to Claude")
        wavs.append(decoded.detach().float().cpu().numpy().squeeze())
        del out, decoded
        torch.cuda.empty_cache()
    return np.concatenate(wavs) if len(wavs) > 1 else wavs[0]

os.makedirs("/content/ablation_audio", exist_ok=True)
print("READY")

In [ ]:
# ===== Build the three data pools =====

def pairs_from_files(files, sigma_filter=None):
    """sigma_filter=None -> all frames (pool A style). sigma_filter=0.0 ->
    only frames where the recorded sigma is exactly 0 (pool B). v1 files
    have no sigma field at all -- treated as all-clean by construction
    (pool C), sigma_filter is ignored for them."""
    hiddens, latents = [], []
    for f in files:
        utt = load_utterance(f)
        if sigma_filter is not None and utt.sigma is not None:
            mask = (utt.sigma.float() == sigma_filter)
            if mask.sum() == 0:
                continue
            hiddens.append(utt.hidden[mask].float())
            latents.append(utt.latent[mask].float())
        else:
            hiddens.append(utt.hidden.float())
            latents.append(utt.latent.float())
    hidden = torch.cat(hiddens)
    latent = torch.cat(latents)
    mean = latent.mean(dim=0)
    std = latent.std(dim=0).clamp_min(1e-4)
    return PairData(hidden=hidden, latent=(latent - mean) / std, mean=mean, std=std)

v2_files = sorted(glob.glob(f"{CACHE_V2_DIR}/*.pt"))
v1_files = sorted(glob.glob(f"{CACHE_V1_DIR}/*.pt"))
print(f"v2: {len(v2_files)} scripts, v1: {len(v1_files)} utterances")

pool_A = pairs_from_files(v2_files, sigma_filter=None)
print(f"pool A (v2, mixed): {pool_A.hidden.shape[0]} frames")

pool_B = pairs_from_files(v2_files, sigma_filter=0.0)
print(f"pool B (v2, sigma==0 only): {pool_B.hidden.shape[0]} frames")

# v1 has 10,000 individual files -- reading them straight off the Drive
# mount one at a time (as pool A/B do fine at 248 files) took all night and
# never finished. The original P1 gate notebook hit this exact problem and
# worked around it with a bulk local copy first; do the same here.
print("copying v1 cache locally first (Drive is slow for many small individual reads)...")
t0 = time.time()
!mkdir -p /content/cache_v1_local && cp {CACHE_V1_DIR}/*.pt /content/cache_v1_local/
v1_files_local = sorted(glob.glob("/content/cache_v1_local/*.pt"))
print(f"copied {len(v1_files_local)} files locally in {time.time()-t0:.0f}s")

pool_C = pairs_from_files(v1_files_local, sigma_filter=None)
print(f"pool C (v1, all clean): {pool_C.hidden.shape[0]} frames")

In [ ]:
# ===== Train (or reuse) the three checkpoints =====

def get_or_train(tag, data):
    path = f"{CKPT_DIR}/{tag}"
    if os.path.exists(path):
        print(f"{tag}: reusing existing checkpoint on Drive, no retrain")
        return load_checkpoint(path)
    print(f"{tag}: training fresh (5K steps)...")
    head = FlowHead(FlowHeadConfig(d_model=data.d_model, d_latent=data.d_latent))
    out = train(head, data, steps=5000, batch_size=1024, lr=2e-4,
               ema_decay=0.999, device="cuda", log_every=500)
    save_checkpoint(path, head, out["ema"], data, step=5000)
    return load_checkpoint(path)

head_A, mean_A, std_A = get_or_train("gate_v2_5k.pt", pool_A)
head_B, mean_B, std_B = get_or_train("gate_v2_clean5k.pt", pool_B)
head_C, mean_C, std_C = get_or_train("gate_5k.pt", pool_C)
for h in (head_A, head_B, head_C):
    h.to("cuda")
print("all three checkpoints ready")

In [ ]:
# ===== Render all three on the SAME short + long test utterances -- LISTEN =====
short_file = sorted(glob.glob(f"{CACHE_V2_DIR}/cv2_150w_*.pt"))[0]
long_file = sorted(glob.glob(f"{CACHE_V2_DIR}/cv2_2400w_*.pt"))[0]

CHECKPOINTS = [
    ("A_mixed", head_A, mean_A, std_A),
    ("B_clean_only", head_B, mean_B, std_B),
    ("C_v1_cache", head_C, mean_C, std_C),
]

for label, test_file in (("short", short_file), ("long", long_file)):
    utt = load_utterance(test_file)
    print(f"=== {label}: {utt.utt_id} ({utt.hidden.shape[0]} frames) | {utt.text[:70]} ===")
    for tag, head, mean, std in CHECKPOINTS:
        z = sample_latents(head, utt.hidden.float(), mean, std, nfe=4)
        wav = decode_latents(z)
        path = f"/content/ablation_audio/{label}_{tag}.wav"
        sf.write(path, wav, 24000)
        print(f"  {tag}")
        display(Audio(path))

In [ ]:
# ===== Bundle for download =====
import zipfile
with zipfile.ZipFile("/content/ablation_audio.zip", "w") as z:
    for f in os.listdir("/content/ablation_audio"):
        z.write(f"/content/ablation_audio/{f}", f)
from google.colab import files as colab_files
colab_files.download("/content/ablation_audio.zip")